# Start Here: Full Project Smoke Test

**Project:** Automated Equity Research Platform

This notebook runs independently from the other notebooks.

## Standalone setup

This notebook is designed to run by itself. It can use a local clone, a Google Drive folder, or a GitHub repository.

Before publishing, replace `YOUR_USERNAME` in `REPO_URL` with your actual GitHub username.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess

PROJECT_NAME = "Automated_Equity_Research_Platform"
REPO_URL = "https://github.com/YOUR_USERNAME/Automated_Equity_Research_Platform.git"

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception:
        pass

candidate_roots = [
    Path.cwd(),
    Path("/content") / PROJECT_NAME,
    Path("/content/drive/MyDrive") / PROJECT_NAME,
    Path("/content/drive/MyDrive/AI_Equity_Research_Platform"),
]

PROJECT_ROOT = next(
    (path for path in candidate_roots if (path / "pyproject.toml").exists()),
    None,
)

if PROJECT_ROOT is None:
    if "YOUR_USERNAME" in REPO_URL:
        raise RuntimeError(
            "Project files were not found. Upload the full project folder to Google Drive "
            "or replace YOUR_USERNAME in REPO_URL with your real GitHub username."
        )
    target = Path("/content") / PROJECT_NAME
    if not target.exists():
        subprocess.run(["git", "clone", REPO_URL, str(target)], check=True)
    PROJECT_ROOT = target

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

print("Project root:", PROJECT_ROOT)

In [ ]:
%pip install -q -r requirements.txt
%pip install -q -e .

In [ ]:
import os

try:
    from google.colab import userdata
    for secret_name in ("SEC_USER_AGENT", "FRED_API_KEY"):
        try:
            value = userdata.get(secret_name)
            if value:
                os.environ[secret_name] = value
        except Exception:
            pass
except Exception:
    pass

print("SEC enabled:", bool(os.getenv("SEC_USER_AGENT")))
print("FRED enabled:", bool(os.getenv("FRED_API_KEY")))

Run a complete project test with user-selected ticker and peers.

In [ ]:
from equity_research import ProjectConfig, EquityResearchPipeline
from equity_research.valuation import DCFInputs
from equity_research.visualization import price_dashboard, ratio_chart, dcf_projection_chart, sensitivity_heatmap

TICKER = input("Ticker to test, for example AAPL: ").strip().upper() or "AAPL"
peer_input = input("Peers separated by commas, or blank for defaults: ").strip()
PEERS = [p.strip().upper() for p in peer_input.split(",") if p.strip()] or ["MSFT", "GOOGL", "AMZN", "META"]

config = ProjectConfig(project_root=PROJECT_ROOT, start_date="2018-01-01", benchmark="SPY", risk_free_rate=0.04)
pipeline = EquityResearchPipeline(config)

result = pipeline.run(
    TICKER,
    peers=PEERS,
    dcf_inputs=DCFInputs(initial_growth=0.08, discount_rate=0.09, terminal_growth=0.025),
    include_sec=bool(os.getenv("SEC_USER_AGENT")),
    include_macro=bool(os.getenv("FRED_API_KEY")),
    persist=True,
)

print("Completed:", result.ticker)
print("Report folder:", config.reports_dir)

In [ ]:
import pandas as pd
display(pd.Series({
    "Company": result.bundle.info.get("longName", result.ticker),
    "Quality Score": result.fundamentals.quality_score,
    "Current Price": result.fundamentals.values.get("price"),
    "DCF Value / Share": result.dcf.intrinsic_value_per_share if result.dcf else None,
    "DCF Upside / Downside": result.dcf.upside_downside if result.dcf else None,
    "Sharpe Ratio": result.risk_metrics.get("sharpe_ratio"),
    "Max Drawdown": result.risk_metrics.get("max_drawdown"),
}).to_frame("Value"))

price_dashboard(result.features.tail(1250), result.ticker).show()
ratio_chart(result.fundamentals.ratios).show()

if result.dcf:
    dcf_projection_chart(result.dcf).show()
    sensitivity_heatmap(result.dcf_sensitivity, result.ticker).show()